# Chạy toàn bộ 3 thí nghiệm

Dự báo log return phiên kế tiếp, sau đó suy ra giá và hướng. Chạy các ô từ trên xuống. Dữ liệu dừng tại 30/09/2025.

## Chuẩn bị môi trường

Trên máy cá nhân, cài `pip install -r requirements.txt` trước. Các lệnh mặc định dùng snapshot được lưu trong dự án; không tự thay dữ liệu bằng lần tải mạng mới.

In [ ]:
# Tìm thư mục gốc khi chạy từ repo, từ notebooks/ hoặc trên Google Colab.
from pathlib import Path
import sys, subprocess, json, os
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/fpt-stock-trends-prediction')]
ROOT = next((p for p in candidates if (p/'src/train.py').exists()), None)
if ROOT is None and 'google.colab' in sys.modules:
    subprocess.run(['git','clone','https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git','/content/fpt-stock-trends-prediction'], check=True)
    ROOT = Path('/content/fpt-stock-trends-prediction')
if ROOT is None:
    raise FileNotFoundError('Hãy mở notebook trong repository đã tải về.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'src'))
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import pandas as pd
from IPython.display import display, Image
print('Thư mục làm việc:', ROOT)


## 1. Xác minh và căn thời điểm dữ liệu

Kiểm tra snapshot, chuẩn hóa nguồn bổ trợ và kiểm toán OHLCV. Không điền giá trị trước ngày có thể biết.

In [ ]:
from fetch_data import main as fetch
assert fetch(['--verify']) == 0
from fetch_supplemental import run_from_cache, RAW_DIR, DEFAULT_OUTPUT
audit = run_from_cache(RAW_DIR, DEFAULT_OUTPUT)
from eda import run_eda
run_eda(ROOT/'data/raw/fpt/fpt_historical_price.csv',ROOT/'figures',ROOT/'reports');

## 2. Tạo đặc trưng và xem chuỗi thời gian

Giữ 32/38/43 biến trên cùng ngày. ADF và KPSS có giả thuyết gốc khác nhau; đọc kết hợp thay vì coi một p-value là bằng chứng chắc chắn.

In [ ]:
from prepare import prepare, DEFAULT_INPUT, DEFAULT_OUTPUT
prepare(DEFAULT_INPUT, DEFAULT_OUTPUT)
from time_series_analysis import run_analysis
run_analysis();
display(Image(filename=str(ROOT/'figures/12_acf_pacf.png')))

## 3. Huấn luyện và đánh giá

Đây là ô tốn thời gian nhất: 5 fold trong Train, rồi 9 cấu hình Test dùng lại cho 3 thí nghiệm. Bảng chính chỉ có 5 phương pháp. Không dùng Test để chọn L hoặc siêu tham số.

In [ ]:
from train import run_training
summary = run_training()
print(json.dumps(summary['selected_model'],ensure_ascii=False,indent=2))
for name in ['main_benchmark','ablation','statistical_extension','lookback']:
    print(name)
    display(pd.read_csv(ROOT/f'reports/{name}.csv'))

## 4. Dự báo và giải thích

Mô hình được chọn qua CV học lại các nhãn đã có để dự báo phiên sau 30/09/2025. Phân tích lỗi và permutation dùng mô hình đánh giá riêng, không dùng bản học lại.

In [ ]:
from predict import predict_latest
predict_latest(ROOT/'data/raw/fpt/fpt_historical_price.csv',ROOT/'models',ROOT/'reports/latest_prediction.json')
from explain import explain
explain();
display(Image(filename=str(ROOT/'figures/permutation_importance.png')))
display(pd.read_csv(ROOT/'reports/error_cases.csv'))

## 5. Kiểm thử

Kiểm tra phép đổi giá, dự báo bằng 0, cửa sổ liên tiếp, căn ngày công bố và tính nhất quán của artefact.

In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q'],check=True)